# Crypto Portfolio Optimization V6 (Binance + RWA cổ phiếu) — Walk-forward

Cách dùng: **chạy cell thư viện → chỉnh 2 cell PARAMS → chạy tuần tự các cell còn lại.** Mọi tham số backtest nằm trong 2 cell PARAMS.

Khác V3 (V3 giữ nguyên, không sửa):
- **Walk-forward**: tại mỗi candle `t`, fit trên `n_window` candle liền trước (không gồm `t`) → weight áp cho return của candle `t`. Tải `display_candles + n_window` nến: nửa đầu chỉ làm cửa sổ fit, nửa sau là phần backtest hiển thị.
- **Hard-rule trễ dữ liệu 1 ngày** (`data_lag_days`): hôm nay là T chỉ có data đến T-1, weight tối ưu từ đó chỉ ảnh hưởng từ T+1.
- Không `bfill` (chặn look-ahead); coin chưa đủ dữ liệu trong cửa sổ bị loại khỏi cửa sổ đó.
- Chọn tài sản / tối ưu / xếp hạng chỉ dùng dữ liệu cửa sổ. Grid `max_assets` vẫn có nhưng mỗi giá trị được walk-forward riêng.
- `mu` arithmetic (không ghép lãi theo giờ) và co về trung bình chéo (`mu_shrink`).
- Một `RF` cho optimizer + metrics; có phí giao dịch trên turnover.

Khác V4 (V4 giữ nguyên, không sửa) — **vũ trụ 100% từ Binance, thêm token cổ phiếu (bStocks)**:
- Bỏ CoinGecko. Vốn hóa = giá x circulating supply từ endpoint web công khai (không chính thức) của Binance; token cổ phiếu nhận diện bằng tag `bStocks`, ETF đòn bẩy/ngược bị loại theo tên sản phẩm.
- Hai nhóm xếp riêng (coin, cổ phiếu): top `candidate_pool` theo vốn hóa -> xếp hạng **Sortino** (daily, ngưỡng RF) -> giữ top `top_n_coins` / `top_n_stocks`. Mặc định KHÔNG yêu cầu đủ lịch sử.
- Mã thiếu lịch sử được **ffill + bfill** để vào được cửa sổ fit; mã có đoạn giá điền được gắn nhãn `*` trên chart/bảng weight và liệt kê % nến điền ở cell tải dữ liệu.
- Lưu ý thiên lệch: (1) bước xếp hạng Sortino dùng dữ liệu tới hôm nay nên có look-ahead so với backtest; (2) giá bfill phẳng làm volatility/mu của mã đó thấp giả tạo trong cửa sổ fit.

In [ ]:
!pip install --upgrade binance-historical-data PyPortfolioOpt -q

In [ ]:
import time
start_time = time.time()

import os
import glob
import re
import requests
import zipfile
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from math import ceil
from datetime import datetime, timedelta, date
from dateutil.relativedelta import relativedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
from rich.progress import Progress
from rich.console import Console
from typing import List
from pypfopt import EfficientFrontier, CovarianceShrinkage, risk_models

# Tải dữ liệu thô từ Binance + lấy top coin (giữ nguyên từ V3)
def date_range_daily(start: str, end: str):
    start_dt = datetime.strptime(start, "%Y-%m-%d")
    end_dt = datetime.strptime(end, "%Y-%m-%d")
    current = start_dt
    dates = []
    while current <= end_dt:
        dates.append(current.strftime("%Y-%m-%d"))
        current += relativedelta(days=1)
    return dates

def date_range_monthly(start: str, end: str):
    start_dt = datetime.strptime(start, "%Y-%m")
    end_dt = datetime.strptime(end, "%Y-%m")
    current = start_dt
    months = []
    while current <= end_dt:
        months.append(current.strftime("%Y-%m"))
        current += relativedelta(months=1)
    return months

def download_binance_klines_fixed_symbols(
    symbols: List[str],
    intervals: List[str],
    time_period: str,          # "daily" or "monthly"
    start: str,                # "2023-01" or "2023-01-01"
    end: str,                  # "2025-11" or "2025-12-10"
    destination_dir: str = "./binance_data",
    max_workers: int = 15,
    retries: int = 3,
    symbol_start: dict = None  # {symbol: 'YYYY-MM-DD'}: bỏ qua file trước ngày niêm yết của từng mã (tránh hàng trăm request 404)
):
    console = Console()
    base_url = "https://data.binance.vision/data"
    asset_path = "spot"

    if time_period == "daily":
        date_list = date_range_daily(start, end)
        url_template = f"{base_url}/{asset_path}/{time_period}/klines/{{symbol}}/{{interval}}/{{symbol}}-{{interval}}-{{date}}.zip"
        csv_name_template = "{symbol}-{interval}-{date}.csv"
    elif time_period == "monthly":
        date_list = date_range_monthly(start, end)
        url_template = f"{base_url}/{asset_path}/{time_period}/klines/{{symbol}}/{{interval}}/{{symbol}}-{{interval}}-{{date}}.zip"
        csv_name_template = "{symbol}-{interval}-{date}.csv"
    else:
        raise ValueError("time_period must be 'daily' or 'monthly'")

    # Chỉ tải file còn thiếu: đã có CSV (hoặc marker .404 = Binance không có file) thì bỏ qua
    stale_before = ((date.today() - timedelta(days=5)).strftime("%Y-%m-%d") if time_period == "daily"
                    else date.today().strftime("%Y-%m"))
    def _csv_path(s, i, d):
        return os.path.join(destination_dir, asset_path, s, i, csv_name_template.format(symbol=s, interval=i, date=d))
    todo = [(s, i, d) for s in symbols for i in intervals for d in date_list
            if not os.path.exists(_csv_path(s, i, d)) and not os.path.exists(_csv_path(s, i, d) + ".404")
            and d >= (symbol_start or {}).get(s, "")[:len(d)]]
    total_files = len(todo)
    if not todo:
        console.print("[bold green]Đã đủ data, bỏ qua bước tải.[/]")
        return
    success_count = 0

    def download_one(symbol: str, interval: str, date_str: str):
        nonlocal success_count
        url = url_template.format(symbol=symbol, interval=interval, date=date_str)
        csv_name = csv_name_template.format(symbol=symbol, interval=interval, date=date_str)
        local_dir = os.path.join(destination_dir, asset_path, symbol, interval)
        local_csv = os.path.join(local_dir, csv_name)
        local_zip = local_csv.replace(".csv", ".zip")

        os.makedirs(local_dir, exist_ok=True)

        # Skip if already have CSV
        if os.path.exists(local_csv):
            return True

        for attempt in range(retries + 1):
            try:
                resp = requests.get(url, timeout=20)
                if resp.status_code == 404:
                    if date_str < stale_before:       # ngày đã cũ mà vẫn 404 = không tồn tại -> đánh dấu, lần sau khỏi hỏi lại
                        open(local_csv + ".404", "w").close()
                    return False
                resp.raise_for_status()

                # Save ZIP
                with open(local_zip, "wb") as f:
                    f.write(resp.content)

                # Extract first CSV
                with zipfile.ZipFile(local_zip) as zf:
                    for name in zf.namelist():
                        if name.endswith(".csv"):
                            zf.extract(name, local_dir)
                            os.rename(os.path.join(local_dir, name), local_csv)
                            break

                os.remove(local_zip)  # optional: save disk space
                return True

            except Exception as e:
                if attempt == retries:
                    console.print(f"[red]Failed {url}: {e}[/]")
                    return False
        return False

    console.print(f"[bold blue]Total files to attempt: {total_files}[/]")

    with Progress() as progress:
        task = progress.add_task("[cyan]Downloading...", total=total_files)
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = []
            for symbol, interval, d in todo:
                futures.append(executor.submit(download_one, symbol, interval, d))

            for future in as_completed(futures):
                if future.result():
                    success_count += 1
                progress.advance(task)

    console.print(f"[bold green]Done. Successfully downloaded {success_count}/{total_files} files.[/]")

DATA_VARS = ("price_df", "dd_hl", "LABEL")                         # sinh ra ở cell tải dữ liệu
RESULT_VARS = ("returns_wf", "weights_hist", "summary")   # sinh ra ở cell walk-forward

def reset_state(*names):
    """Xóa biến của các bước phía sau khi cell tham số/dữ liệu được chạy lại -> chạy lẻ cell sau sẽ báo NameError
    thay vì âm thầm dùng kết quả của lần chạy trước (không khớp tham số mới)."""
    for n in names:
        globals().pop(n, None)

def detect_timestamp_unit(timestamp):
    num_digits = len(str(timestamp))
    if num_digits == 13:
        return 'ms'
    elif num_digits == 16:
        return 'us'
    else:
        raise ValueError(f"Timestamp không hợp lệ: {timestamp}")

def convert_timestamp(timestamp):
    unit = detect_timestamp_unit(timestamp)
    return pd.to_datetime(timestamp, unit=unit, errors='coerce')

def read_csv_file(file_path):
    # File Binance KHÔNG có dòng header (V3 dùng header mặc định nên mất dòng đầu mỗi file = nến 00:00).
    df = pd.read_csv(file_path, header=None)
    df = df[pd.to_numeric(df[0], errors="coerce").notna()].astype({0: "int64", 6: "int64"})   # bỏ header nếu có
    df.columns = [
        "open_time", "Open", "High", "Low", "Close", "volume",
        "close_time", "quote_asset_volume", "number_of_trades",
        "taker_buy_base_asset_volume", "taker_buy_quote_asset_volume", "ignore"
    ]
    if df.empty:
        return df
    unit = 'ms' if len(str(df['open_time'].iloc[0])) == 13 else 'us'   # vectorize: nhanh hơn apply từng dòng
    df['open_time'] = pd.to_datetime(df['open_time'], unit=unit)
    df['close_time'] = pd.to_datetime(df['close_time'], unit=unit)
    return df

BN_WEB = "https://www.binance.com/bapi/asset/v2/public/asset-service/product/get-products"
LEVERAGED = r"(?:\d[xX]\b|Bull|Bear|Ultra|Inverse|VIX|Leveraged)"      # tên ETF đòn bẩy / ngược / biến động
COIN_BLACKLIST = {
    "USDT", "USDC", "DAI", "TUSD", "UST", "WBTC", "STETH", "FIGR_HELOC", "WBT", "WSTETH",
    "WBETH", "USDS", "LEO", "BSC-USD", "WETH", "FDUSD", "USDP", "GUSD",
    "WEETH", "XMR", "USDE", "CBBTC", "SUSDS", "CRO", "USDT0", "PYUSD", "SUSDE", "HASH",
    "MNT", "USD1", "CC", "BGB", "OKB", "M", "USDF", "BUIDL", "JITOSOL",
    "PI", "RAIN", "JLP", "HTX", "SYRUPUSDC", "USYC", "USDG", "JSOL",
    "KCS", "RLUSD", "RSETH", "BNSOL", "WBNB", "BFUSD", "RIVER", "GT", "USDD", "RETH",
    "MYX", "KAS", "LBTC", "FBTC", "USDTB", "USTB", "USDC.E", "FLR", "SYRUPUSDT", "METH",
    "WM", "JAAA", "LSETH", "KHYPE", "USDY", "OUSG", "XDC", "SOLVBTC", "EUTBL", "USDAI",
    "COCA", "CLBTC", "BDX", "WFLR", "USD0", "JTRSY", "OSETH", "A7A5", "GHO", "EZETH",
    "IP", "JUPSOL", "STABLE", "FTN", "TBTC", "CTM", "EURC", "USR", "PIPPIN", "KAU",
    "YLDS", "SIREN"
}

def binance_products(quote: str = "USDT") -> pd.DataFrame:
    """Sản phẩm spot đang TRADING của Binance với cặp {quote} (index = base asset): giá, circulating supply `cs`,
    vốn hóa mcap = giá x cs, nhóm (stock = tag bStocks, commodity = tCommodities, leveraged = ETF đòn bẩy/ngược).
    Dùng endpoint web công khai (không chính thức) vì API chính thức không có vốn hóa/tag; lỗi thì dừng, không tự đổi nguồn."""
    r = requests.get(BN_WEB, params={"includeEtf": "true"}, headers={"User-Agent": "Mozilla/5.0"}, timeout=30)
    r.raise_for_status()
    df = pd.DataFrame(r.json()["data"])
    df = df[(df.q == quote) & (df.st == "TRADING")].copy()
    for c in ("c", "cs"):
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df["mcap"] = df.c * df.cs
    df["stock"] = df.tags.apply(lambda t: "bStocks" in (t or []))
    df["commodity"] = df.tags.apply(lambda t: "tCommodities" in (t or []))
    df["leveraged"] = df.stock & df.an.str.contains(LEVERAGED, case=False, regex=True)
    return df.set_index("b")

def get_top_coins(n_coins: int = 500, kind: str = "crypto", quote: str = "USDT", exclude_leveraged: bool = True) -> list:
    """
    Top N theo vốn hóa Binance của nhóm `kind`: 'crypto' (bỏ stablecoin / token wrapped / hàng hóa trong COIN_BLACKLIST)
    hoặc 'stock' (token cổ phiếu bStocks, bỏ ETF đòn bẩy/ngược nếu exclude_leveraged).
    Trả List[str] base asset viết hoa, không trùng lặp.
    """
    p = binance_products(quote)
    p = p[p.mcap > 0]
    if kind == "stock":
        p = p[p.stock & ~(p.leveraged & exclude_leveraged)]
    else:
        p = p[~p.stock & ~p.commodity & ~p.index.str.upper().isin(COIN_BLACKLIST)]
    return list(dict.fromkeys(p.sort_values("mcap", ascending=False).index.str.upper()))[:n_coins]

def get_start_date(
    timeframe: str,
    required_samples: int = 10000,
    end_date: datetime | None = None,
) -> tuple[datetime, int]:
    """
    Calculate start date to obtain at least `required_samples` candles.

    Parameters
    ----------
    timeframe : str
        e.g. '1m', '5m', '15m', '1h', '4h', '1d', '1w'
    required_samples : int
        Minimum number of candles required.
    end_date : datetime
        End date. Defaults to current UTC time.

    Returns
    -------
    start_date : datetime
    required_days : int
    """
    if end_date is None:
        end_date = datetime.utcnow()

    unit = timeframe[-1].lower()
    value = int(timeframe[:-1])

    if unit == "m":
        candles_per_day = 24 * 60 / value

    elif unit == "h":
        candles_per_day = 24 / value

    elif unit == "d":
        candles_per_day = 1 / value

    elif unit == "w":
        candles_per_day = 1 / (7 * value)

    else:
        raise ValueError(f"Unsupported timeframe: {timeframe}")

    required_days = ceil(required_samples / candles_per_day)
    start_date = end_date - timedelta(days=required_days)

    return start_date, required_days

import warnings
from joblib import Parallel, delayed, effective_n_jobs

# ====== Thư viện V4: walk-forward, không look-ahead (RF lấy từ block params) ======
def tf_delta(tf: str) -> pd.Timedelta:
    return pd.Timedelta(tf.replace('m', 'min'))


def periods_per_year(tf: str) -> float:
    return 365 * 24 * 3600 / tf_delta(tf).total_seconds()


def load_prices(tickers, start, end, timeframe, data_dir="binance_data", col="Close", bfill=False, first_real=None):
    """Close price của mọi ticker trên 1 index chung. ffill (causal). bfill=True: điền cả đoạn TRƯỚC khi niêm yết bằng giá thật đầu tiên
    (mã vào được cửa sổ fit nhưng đoạn đó phẳng và dùng giá tương lai -> gắn nhãn, đừng coi là dữ liệu thật). bfill=False: giữ NaN.
    `first_real` (dict, tùy chọn) nhận {ticker: timestamp nến thật đầu tiên} TRƯỚC khi điền."""
    delta = tf_delta(timeframe)
    idx = pd.date_range(pd.Timestamp(start, tz="UTC"),
                        pd.Timestamp(end, tz="UTC") + pd.Timedelta(days=1) - delta, freq=delta)
    def load_one(t):
        files = glob.glob(os.path.join(data_dir, "spot", t, timeframe, "*.csv"))  # mỗi file đọc 1 lần
        if not files:
            print(f"⚠️  Bỏ {t}: không có file")
            return None
        dfs = [d for d in (read_csv_file(f)[["open_time", col]].dropna() for f in files) if not d.empty]
        if not dfs:                               # bỏ file rỗng/all-NA trước khi concat (tránh FutureWarning)
            print(f"⚠️  Bỏ {t}: file rỗng")
            return None
        df = pd.concat(dfs)
        df["open_time"] = pd.to_datetime(df["open_time"], utc=True)
        s = df.drop_duplicates("open_time").set_index("open_time")[col]
        return s[s > 0].reindex(idx)

    # ponytail: tuần tự; thử ThreadPool không nhanh hơn (GIL), cần nữa thì dùng joblib theo mã
    cols = {t: s for t, s in zip(tickers, map(load_one, tickers)) if s is not None}
    df = pd.DataFrame(cols)
    if first_real is not None:
        first_real.update({t: df[t].first_valid_index() for t in df})
    return df.ffill().bfill() if bfill else df.ffill()


def cov_matrix(win, P, model, span):
    shrink = lambda: CovarianceShrinkage(win, frequency=P).ledoit_wolf()
    if model == "shrinkage":
        return shrink()
    if model == "sample":
        return risk_models.sample_cov(win, frequency=P)
    S = risk_models.exp_cov(win, frequency=P, span=span)
    return S if model == "exp" else 0.5 * S + 0.5 * shrink()


def max_drawdown(r: pd.Series) -> float:
    cum = (1 + r).cumprod()
    return (cum / cum.cummax() - 1).min()


def optimize_window(win, rets, P, cfg, max_assets):
    """Chọn tài sản + tối ưu CHỈ trên `win` (dữ liệu quá khứ). Trả {strategy: weights}."""
    mu = rets.mean() * P                      # arithmetic, không ghép lãi theo giờ
    sd = rets.std() * np.sqrt(P)
    ranked = (mu / sd).sort_values(ascending=False).index

    req = {a: m for a, m in cfg["required_assets"].items() if a in win.columns}
    sel = list(dict.fromkeys(list(req) + list(ranked)))[:max_assets]
    lo = [req.get(a, cfg["min_weight"]) for a in sel]
    hi = [cfg["eur_max_weight"] if a in cfg["eur_assets"]
          else 1.0 if a in req else cfg["other_max_weight"] for a in sel]   # mã không bắt buộc bị chặn trần
    if sum(lo) > 1 or sum(hi) < 1 or any(l > h for l, h in zip(lo, hi)):    # không khả thi -> bỏ bước này
        return {}

    s = cfg["mu_shrink"]                      # co mu về trung bình chéo -> bớt nhiễu
    mu_s = (1 - s) * mu[sel] + s * mu[sel].mean()
    S = cov_matrix(win[sel], P, cfg["risk_model"], cfg["span"])
    bounds = list(zip(lo, hi))

    def solve(kind):
        ef = EfficientFrontier(mu_s, S, weight_bounds=bounds, solver_options={"adaptive_rho_interval": 25})
        try:
            if kind == "sharpe":
                ef.max_sharpe(risk_free_rate=RF)
            elif kind == "optimal":
                ef.efficient_return(cfg["target_return"])
            else:
                ef.min_volatility()
        except Exception:                     # vd: mọi mu < RF, target vượt khả thi
            ef = EfficientFrontier(mu_s, S, weight_bounds=bounds, solver_options={"adaptive_rho_interval": 25})
            ef.min_volatility()
        return dict(ef.clean_weights())

    return {k: solve(k) for k in ("sharpe", "volatility", "optimal")}


def round_weights(w, step):
    """Làm tròn weight về bội số của `step` (largest remainder): tổng giữ nguyên, không có số lẻ hơn `step`.
    Sàn/trần là bội của step nên vẫn được giữ."""
    if not step or not w:
        return w
    names = list(w)
    x = np.array([w[n] for n in names]) / step
    units = int(round(x.sum()))
    base = np.floor(x + 1e-9).astype(int)
    for i in np.argsort(-(x - base))[:max(0, units - base.sum())]:
        base[i] += 1
    return {n: float(b * step) for n, b in zip(names, base)}


def bench_universe(rets, P, cfg):
    """Mã dùng cho benchmark (equal_weight, risk_parity, absolute_equal_dd): các mã gốc trong required_assets + (benchmark_n_assets - số mã gốc)
    mã có Sortino cao nhất trong cửa sổ fit (ngưỡng RF, chỉ dữ liệu cửa sổ). Giữ tổng số mã <= benchmark_n_assets (= max_assets_limit)."""
    base = [a for a in cfg["required_assets"] if a in rets.columns]
    r = rets.drop(columns=base)
    dd = np.sqrt((np.minimum(r - RF / P, 0) ** 2).mean()).replace(0, np.nan)
    top = ((r.mean() - RF / P) / dd).nlargest(max(0, cfg["benchmark_n_assets"] - len(base))).index
    return base + list(top)


def make_targets(win, P, cfg, max_assets_list, mdd_hist=None):
    """Target weight tại 1 mốc rebalance. Benchmark không phụ thuộc max_assets -> tính 1 lần;
    opt_*_m{m} tính riêng cho từng m trong max_assets_list."""
    rets = win.pct_change().dropna()
    brets = rets[bench_universe(rets, P, cfg)]    # benchmark chỉ dùng tập mã giới hạn (không vượt max_assets_limit)
    cols = list(brets.columns)
    out = {"equal_weight": {c: 1 / len(cols) for c in cols}}

    inv = 1 / brets.std().replace(0, np.nan).dropna()
    out["risk_parity"] = (inv / inv.sum()).to_dict()

    # equal absolute drawdown: phần dư (cash_asset) không tham gia ngân sách rủi ro
    cash = cfg["cash_asset"]
    mdds = {c: abs(max_drawdown(brets[c])) for c in cols if c != cash}
    if mdd_hist:                              # MDD lịch sử dài (vd 5 năm) thay cho MDD của cửa sổ fit
        mdds = {c: mdd_hist.get(c, m) for c, m in mdds.items()}
    mdds = {c: m for c, m in mdds.items() if m > 0}
    for tgt in map(abs, cfg["mdd_targets"]):
        if not mdds:
            break
        w = {c: (tgt / len(mdds)) / m for c, m in mdds.items()}
        tot = sum(w.values())
        if tot > 1:
            w = {c: x / tot for c, x in w.items()}
            tot = 1.0
        if cash in cols:
            w[cash] = 1 - tot                 # nếu thiếu cash_asset: phần dư để tiền mặt (0%)
        out[f"absolute_equal_dd_{int(tgt * 100)}"] = w

    for m in max_assets_list:
        try:
            out.update({f"opt_{k}_m{m}": v for k, v in optimize_window(win, rets, P, cfg, m).items()})
        except Exception as e:
            print(f"  ⚠️ optimize lỗi (m={m}), giữ weight cũ: {e}")
    return out


def targets_chunk(prices, ts, n_window, timeframe, cfg, max_assets_list, lag, dd_hl):
    """Pha 1 (song song): target của mọi strategy tại các mốc rebalance `ts`. Chỉ phụ thuộc data cửa sổ, không phụ thuộc đường đi."""
    # cvxpy hay báo "Solution may be inaccurate" ở các bước tối ưu biên; kết quả vẫn dùng được -> tắt cho gọn
    warnings.filterwarnings("ignore", message="Solution may be inaccurate", category=UserWarning)
    P = periods_per_year(timeframe)
    out = []
    for t in ts:
        mdd_hist = None
        if dd_hl is not None:                     # MDD từng mã trong N năm gần nhất, chỉ dùng data đã biết
            end = prices.index[t - lag - 1] + tf_delta(timeframe)
            dh, dl = dd_hl                        # High, Low khung mdd_timeframe: đỉnh = cummax(High), đáy = Low
            m = (dh.index + tf_delta(cfg["mdd_timeframe"]) <= end) & (dh.index > end - pd.DateOffset(years=cfg["mdd_years"]))
            dh, dl = dh[m], dl[m]
            mdd_hist = {c: abs((dl[c][dh[c].notna()] / dh[c].dropna().cummax() - 1).min())
                        for c in dh.columns if dh[c].notna().sum() > 1}
        win = prices.iloc[t - lag - n_window:t - lag].dropna(axis=1)
        tg = make_targets(win, P, cfg, max_assets_list, mdd_hist)
        out.append({n: round_weights(w, cfg["weight_step"]) for n, w in tg.items()})
    return out


def limit_move(prev, tgt, cap, step):
    """Dịch weight từ `prev` về `tgt` (cả hai nằm trên lưới `step`, tổng 1) sao cho mỗi mã đổi tối đa `cap` (chuyển theo cặp tăng <-> giảm
    nên tổng giữ nguyên). Mỗi mã đi đơn điệu từ prev tới tgt nên sàn/trần weight vẫn được giữ. Mã chưa chuyển hết thì tiếp ở lần rebalance sau."""
    r = np.rint((tgt - prev) / step).astype(int)            # cần đổi (đơn vị step), có dấu
    a = np.full(len(r), max(1, int(round(cap / step))))     # hạn mức còn lại mỗi mã
    out = np.zeros(len(r), int)
    while True:
        pos, neg = np.flatnonzero((r > 0) & (a > 0)), np.flatnonzero((r < 0) & (a > 0))
        if not len(pos) or not len(neg):
            break
        i, j = pos[np.argmax(r[pos])], neg[np.argmin(r[neg])]
        m = min(r[i], -r[j], a[i], a[j])
        out[i] += m; out[j] -= m; r[i] -= m; r[j] += m; a[i] -= m; a[j] -= m
    return np.round(prev + out * step, 10)


def walk_forward(prices, n_window, timeframe, cfg, max_assets_list, rebalance_every=1, fee=0.001, threshold=0.0, lag=0, dd_hl=None, n_jobs=-1, max_move=None):
    """Tại candle t: fit trên prices[t-lag-n_window : t-lag] -> áp weight cho return của candle t.
    `lag` (candle) = độ trễ dữ liệu: tại thời điểm t chỉ biết giá đến t-lag (hard-rule: dữ liệu luôn trễ 1 ngày).
    Giữa 2 lần rebalance weight trôi theo giá. Phí = fee * turnover.
    Tại mỗi mốc rebalance, strategy chỉ giao dịch nếu weight lệch >= threshold (max |target - hiện tại|).
    `max_move` = mức đổi weight tối đa của MỖI mã giữa 2 lần rebalance (vd 0.01 = 1%); lớn hơn thì dịch dần về target qua các lần sau.
    Trả (returns, weights_hist = weight THỰC HIỆN tại các lần giao dịch, n_target = số mã tối đa trong TARGET của từng strategy).
    Pha 1: tính target mọi mốc rebalance song song (benchmark + mọi max_assets dùng chung cửa sổ/MDD).
    Pha 2: vòng candle tuần tự, vector hóa mọi strategy thành ma trận (S x A)."""
    cols = list(prices.columns)
    pos = {c: i for i, c in enumerate(cols)}
    start = n_window + lag
    total = len(prices) - start
    ts = list(range(start, len(prices), rebalance_every))

    t0 = time.time()
    chunks = np.array_split(ts, max(1, min(len(ts), 4 * effective_n_jobs(n_jobs))))
    res = Parallel(n_jobs=n_jobs)(delayed(targets_chunk)(prices, c.tolist(), n_window, timeframe, cfg, max_assets_list, lag, dd_hl) for c in chunks)
    targets_list = [x for r in res for x in r]
    print(f"  pha 1: {len(ts)} mốc rebalance x {len(max_assets_list)} giá trị max_assets - {time.time() - t0:.0f}s")

    hold = "hold_" + cfg["hold_asset"]
    names = list(dict.fromkeys([n for tg in targets_list for n in tg] + ([hold] if cfg["hold_asset"] in pos else [])))
    row = {n: i for i, n in enumerate(names)}
    S, A = len(names), len(cols)
    Wm = np.zeros((S, A))                         # weight hiện tại (đã trôi theo giá) của từng strategy
    live = np.zeros(S, bool)                      # strategy đã có weight lần đầu chưa
    Lm = np.zeros((S, A))                         # weight thực hiện ở lần giao dịch gần nhất (trên lưới weight_step, không trôi theo giá)
    n_target = {}
    if max_move and not cfg["weight_step"]:
        raise ValueError("max_move cần cfg['weight_step'] > 0 (weight nằm trên lưới)")
    Rm = np.full((total, S), np.nan)
    rets = prices.pct_change().fillna(0).to_numpy()
    hist = {}
    targets_it = iter(targets_list)

    for k, t in enumerate(range(start, len(prices))):
        turn = np.zeros(S)
        if k % rebalance_every == 0:
            targets = next(targets_it)
            if k == 0 and cfg["hold_asset"] in pos:
                targets[hold] = {cfg["hold_asset"]: 1.0}   # buy & hold, không rebalance
            for name, tw in targets.items():
                i = row[name]
                vec = np.zeros(A)
                for a, x in tw.items():
                    vec[pos[a]] = x
                if live[i] and np.abs(vec - Wm[i]).max() < threshold:
                    continue                      # lệch chưa đủ -> giữ nguyên, không phí
                n_target[name] = max(n_target.get(name, 0), int((vec > 0).sum()))
                if max_move and live[i]:
                    vec = limit_move(Lm[i], vec, max_move, cfg["weight_step"])   # mỗi mã chỉ đổi tối đa max_move mỗi lần rebalance
                turn[i] = np.abs(vec - Wm[i]).sum()
                Wm[i] = Lm[i] = vec
                live[i] = True
                hist.setdefault(name, {})[prices.index[t]] = pd.Series(vec, index=cols)
        r = rets[t]
        pr = Wm @ r
        Rm[k] = np.where(live, pr - fee * turn, np.nan)
        Wm *= 1 + r
        Wm /= (1 + pr)[:, None]

    returns = pd.DataFrame(Rm, index=prices.index[start:], columns=names).dropna(axis=1, how="all")
    return returns, {n: pd.DataFrame(h).T.fillna(0) for n, h in hist.items()}, n_target


def summarize(returns, timeframe, weights_hist):
    P = periods_per_year(timeframe)
    rows = {}
    for name, r in returns.items():
        eq = (1 + r).cumprod()
        sd = r.std()
        dd = np.sqrt((np.minimum(r - RF / P, 0) ** 2).mean())      # downside deviation, ngưỡng = RF/P (cùng ngưỡng với Sharpe)
        wh = weights_hist.get(name)
        # turnover/năm = tổng |Δweight| giữa các lần rebalance (không tính trôi giá)
        tn = wh.diff().abs().sum().sum() if wh is not None and len(wh) > 1 else 0.0
        rows[name] = {
            "return_cagr": eq.iloc[-1] ** (P / len(r)) - 1,
            "volatility": sd * np.sqrt(P),
            "sharpe": (r.mean() - RF / P) / sd * np.sqrt(P) if sd > 0 else 0.0,
            "sortino": (r.mean() - RF / P) / dd * np.sqrt(P) if dd > 0 else 0.0,
            "max_drawdown": (eq / eq.cummax() - 1).min(),
            "turnover_per_year": tn * P / len(r),
        }
    return pd.DataFrame(rows).T.sort_values("sortino", ascending=False)


def _daily_closes(sym, start_ms, end_ms):
    """Close 1d từ Binance REST (≤1000 nến/request). Series index = ngày UTC, hoặc None nếu không có dữ liệu."""
    try:
        r = requests.get("https://api.binance.com/api/v3/klines", timeout=15,
                         params=dict(symbol=sym, interval="1d", startTime=start_ms, endTime=end_ms, limit=1000))
        d = r.json() if r.status_code == 200 else []
    except Exception:
        return None
    return pd.Series([float(x[4]) for x in d], index=pd.to_datetime([x[0] for x in d], unit="ms")) if d else None

def screen_sortino(symbols, quote, start_date, end_date, top_n, rf, require_full_history=False, min_days=30, workers=8):
    """Xếp hạng Sortino (daily, ngưỡng rf, năm hóa 365) của {symbol}{quote} trên [start_date, end_date] rồi giữ top_n.
    Mã có < min_days ngày dữ liệu bị loại (Sortino quá nhiễu); require_full_history=True loại thêm mã có nến đầu sau start_date + 3 ngày.
    Trả DataFrame (index = symbol) cột sortino, first (ngày nến đầu), days. Dùng dữ liệu tới end_date -> có look-ahead so với backtest."""
    s_ms = int(pd.Timestamp(start_date).timestamp() * 1000)
    e_ms = int((pd.Timestamp(end_date) + pd.Timedelta(days=1)).timestamp() * 1000) - 1
    with ThreadPoolExecutor(max_workers=workers) as ex:
        closes = list(ex.map(lambda sym: _daily_closes(sym + quote, s_ms, e_ms), symbols))
    rows = {}
    for sym, c in zip(symbols, closes):
        if c is None or len(c) < min_days + 1:
            continue
        if require_full_history and c.index[0] > pd.Timestamp(start_date) + pd.Timedelta(days=3):
            continue
        r = c.pct_change().dropna()
        dd = np.sqrt((np.minimum(r - rf / 365, 0) ** 2).mean())
        if dd > 0:
            rows[sym] = dict(sortino=(r.mean() - rf / 365) / dd * np.sqrt(365), first=c.index[0].date(), days=len(c))
    tab = pd.DataFrame(rows).T
    tab["sortino"] = tab["sortino"].astype(float)
    return tab.sort_values("sortino", ascending=False).head(top_n)


In [ ]:
# ====================== PARAMS 1/2: DỮ LIỆU & BACKTEST ======================
reset_state(*DATA_VARS, *RESULT_VARS)       # đổi tham số -> dữ liệu và kết quả cũ không còn hợp lệ, phải chạy lại các cell sau

# --- Khung thời gian & độ dài backtest ---
timeframe = '1h'            # Khung nến Binance: 1m 5m 15m 30m 1h 2h 4h 6h 8h 12h 1d 1w
display_candles = 10000     # Số nến backtest được hiển thị/đánh giá (phần out-of-sample)
n_window = display_candles  # Số nến lịch sử liền trước dùng để fit weight ở mỗi bước.
                            # Tổng nến tải về = display_candles + n_window (mặc định gấp đôi)

# --- Tần suất & chi phí ---
n_jobs = -1                 # Số tiến trình chạy song song tính target (mỗi mốc rebalance chạy độc lập, gồm cả grid max_assets).
                            # -1 = dùng mọi core, 1 = tuần tự, 4 = tối đa 4 core
rebalance_days = 1          # Chu kỳ tính lại weight, tính bằng NGÀY (30 = hàng tháng, 7 = hàng tuần, 1 = hàng ngày).
                            # Tự quy ra số candle theo timeframe (1h -> 720, 1d -> 30). Giữa 2 lần
                            # rebalance weight trôi theo giá. Nhỏ hơn 1 candle -> mỗi candle (chậm nhất)
rebalance_threshold = 0.01  # Chỉ rebalance khi weight lệch >= ngưỡng này (điểm % tuyệt đối của bất kỳ asset nào,
                            # 0.01 = 1%) so với target mới, kiểm tra tại mỗi mốc rebalance_days.
                            # 0 = luôn rebalance; chưa đủ lệch -> giữ nguyên, không tốn phí
data_lag_days = 1           # HARD-RULE độ trễ dữ liệu (ngày): hôm nay là T chỉ có data đến T-1, nên weight tối ưu
                            # từ data đó chỉ ảnh hưởng từ T+1. Quy ra candle theo timeframe. 0 = tắt (chỉ để so sánh)
max_daily_weight_change = 0.01   # Mỗi mã chỉ được tăng/giảm tối đa chừng này weight MỖI NGÀY (0.01 = 1%): target xa hơn thì dịch dần qua các ngày,
                            # tránh sốc 10-20% trong 1 ngày. Tính theo chu kỳ rebalance (rebalance_days ngày -> hạn mức nhân tương ứng).
                            # Lần phân bổ đầu tiên từ tiền mặt không bị giới hạn. 0 = tắt. Cần cfg['weight_step'] > 0
fee = 0.001                 # Phí mỗi lần giao dịch, tính trên turnover (0.001 = 0.1%)
RF = 0.03                   # Lãi suất phi rủi ro / năm, dùng cho max-Sharpe VÀ cho chỉ số Sharpe

# --- Vũ trụ tài sản (100% từ Binance: vốn hóa = giá x circulating supply; token cổ phiếu có tag bStocks) ---
candidate_pool = 500        # Lấy top N theo vốn hóa MỖI nhóm (coin / cổ phiếu) làm ứng viên. Binance chỉ có ~420 coin và ~100 cổ phiếu cặp USDT nên 500 = lấy hết
candidate_frac_coins = 0.20   # Chỉ lấy top X% coin theo vốn hóa làm ứng viên cho bước xếp hạng Sortino (0.20 = 20% đầu; lọc bỏ phần lớn mã vốn hóa nhỏ). Không nhỏ hơn top_n_coins
candidate_frac_stocks = 1.0   # Tương tự cho nhóm cổ phiếu (1.0 = lấy hết)
top_n_coins = 10            # Sau khi xếp hạng Sortino (daily, ngưỡng RF), giữ top N coin
top_n_stocks = 10           # ... và top N token cổ phiếu
screen_min_days = 30        # Mã có ít hơn N ngày dữ liệu bị loại khỏi bước xếp hạng Sortino (quá nhiễu)
exclude_leveraged_etf = True  # Loại ETF đòn bẩy / ngược / VIX khỏi nhóm cổ phiếu
require_full_history = False  # False = chấp nhận mã niêm yết muộn: giá trước ngày niêm yết được ffill/bfill và gắn nhãn *.
                            # True = chỉ giữ mã đã niêm yết trước ngày bắt đầu dữ liệu
                            # LƯU Ý: xếp hạng Sortino dùng dữ liệu tới hôm nay nên có look-ahead so với backtest
quote_asset = 'USDT'        # Đồng định giá; ticker = symbol + quote_asset (vd BTC -> BTCUSDT)
extra_tickers = ['EURUSDT', 'PAXGUSDT', 'BTCUSDT', 'ETHUSDT']   # Tài sản thêm tay, luôn nằm trong vũ trụ, không qua xếp hạng (EUR, vàng + BTC/ETH vì cfg['required_assets'] cần)

# --- Tải dữ liệu ---
data_dir = 'binance_data'   # Thư mục lưu CSV Binance (gitignore)
download_workers = 15       # Số luồng tải song song
download_retries = 3        # Số lần thử lại mỗi file

# --- Dẫn xuất (không cần sửa) ---
max_move = max_daily_weight_change * rebalance_days if max_daily_weight_change else None      # hạn mức đổi weight mỗi mã / lần rebalance
rebalance_every = max(1, round(rebalance_days * 86400 / tf_delta(timeframe).total_seconds()))   # số candle
lag_candles = round(data_lag_days * 86400 / tf_delta(timeframe).total_seconds())              # độ trễ, số candle

In [ ]:
# ====================== PARAMS 2/2: CHIẾN LƯỢC ======================
reset_state(*RESULT_VARS)                   # đổi tham số chiến lược -> kết quả walk-forward cũ không còn hợp lệ

# Grid test: chạy walk-forward riêng cho từng giá trị max_assets rồi so sánh.
max_assets_limit = top_n_coins + top_n_stocks       # Trần số assets trong port: bỏ giá trị m lớn hơn trần, và bỏ mọi strategy có lúc nắm > trần mã
                            # (vd equal_weight trên cả vũ trụ) khỏi kết quả -> không bao giờ được chọn làm "best"
# m bắt đầu từ 4 vì 4 mã gốc (EUR, vàng, BTC, ETH) luôn chiếm slot đầu; chậm theo số giá trị (4..20 = 17 lần). Thu hẹp khi thử nhanh
max_assets_range = range(4, max_assets_limit + 1)

cfg = dict(
    # --- Ràng buộc trọng số ---
    required_assets={'EURUSDT': 0.15, 'PAXGUSDT': 0.09, 'BTCUSDT': 0.06, 'ETHUSDT': 0.04},
                            # Mã bắt buộc có mặt + trọng số TỐI THIỂU. Tổng sàn phải <= 100%
    min_weight=0.01,        # Trọng số tối thiểu của các mã còn lại được chọn
    other_max_weight=0.10,  # Trọng số TỐI ĐA của mã không nằm trong required_assets (áp cho các strategy opt_*).
                            # Nếu tổng trần không đủ 100% (quá ít mã bắt buộc) thì bước tối ưu đó bị bỏ qua
    eur_assets={'EURUSDT', 'EURIUSDT'},   # Nhóm tài sản bị chặn trần bởi eur_max_weight
    eur_max_weight=0.34,    # Trọng số tối đa cho mỗi mã trong eur_assets

    # --- Mục tiêu tối ưu ---
    target_return=0.05,     # Return/năm mục tiêu cho strategy 'opt_optimal' (efficient_return).
                            # Không đạt được -> tự chuyển về min-volatility

    # --- Ước lượng đầu vào cho optimizer ---
    risk_model='exp_shrink',  # Ma trận hiệp phương sai: 'shrinkage' (Ledoit-Wolf) | 'sample' |
                              # 'exp' (EWMA) | 'exp_shrink' (50% EWMA + 50% Ledoit-Wolf)
    cov_span_days=60,       # Span EWMA tính theo ngày (chỉ dùng cho 'exp'/'exp_shrink')
    mu_shrink=0.5,          # Co return kỳ vọng về trung bình chéo: 0 = giữ nguyên mu thô,
                            # 1 = mọi mã cùng mu. Tăng để bớt nhiễu / bớt đuổi theo mã thắng

    # --- Làm tròn weight ---
    weight_step=0.01,       # Mọi target weight là bội số của bước này (0.01 = 1%, không số lẻ hơn). 0 = tắt

    # --- Benchmark ---
    mdd_targets=(-0.08, -0.13, -0.21, -0.34, -0.55),
                            # MDD mục tiêu cho các danh mục absolute_equal_dd_xx
    mdd_years=5,            # absolute_equal_dd lấy MDD từng mã trong N năm gần nhất (rolling, chỉ data đã biết)
                            # thay vì chỉ trong cửa sổ fit; mã niêm yết ngắn hơn N năm dùng toàn bộ lịch sử có
    mdd_timeframe='1d',     # Khung nến dùng tính MDD dài hạn (tải riêng, nhẹ hơn nhiều so với nến 1h).
                            # MDD = (Low - cummax(High)) / cummax(High): bắt cả đợt sụt trong ngày
    cash_asset='EURUSDT',   # Mã nhận phần dư của absolute_equal_dd (không tham gia ngân sách rủi ro)
    hold_asset='BTCUSDT',   # Mã dùng làm benchmark buy & hold
    benchmark_n_assets=max_assets_limit,
                            # Benchmark (equal_weight, risk_parity, absolute_equal_dd) chỉ dùng N mã: các mã gốc trong required_assets
                            # (EUR, vàng, BTC, ETH) + (N - số mã gốc) mã có Sortino cao nhất trong cửa sổ fit -> không vượt max_assets_limit
)


In [ ]:
# --- Tải & nạp dữ liệu ---
t_load = time.time()
reset_state(*DATA_VARS, *RESULT_VARS)
end_date = date.today() - timedelta(days=1)
start_date, days = get_start_date(timeframe, display_candles + n_window + lag_candles + 24, end_date)   # +24 nến đệm
start_str, end_str = start_date.strftime("%Y-%m-%d"), end_date.strftime("%Y-%m-%d")
print(f"{timeframe} -> {days} days: {start_str} .. {end_str}")

# Vũ trụ: top vốn hóa Binance (coin và cổ phiếu xếp riêng) -> xếp hạng Sortino -> top N mỗi nhóm
extra_bases = {t.removesuffix(quote_asset) for t in extra_tickers}
screens = []
for kind, top_n, frac in (("crypto", top_n_coins, candidate_frac_coins), ("stock", top_n_stocks, candidate_frac_stocks)):
    pool = [b for b in get_top_coins(candidate_pool, kind, quote_asset, exclude_leveraged_etf) if b not in extra_bases]
    pool = pool[:max(top_n, ceil(len(pool) * frac))]       # chỉ giữ top X% theo vốn hóa (get_top_coins đã sắp theo vốn hóa giảm dần)
    tab = screen_sortino(pool, quote_asset, start_date, end_date, top_n, RF, require_full_history, screen_min_days)
    tab.insert(0, "group", kind)
    print(f"{kind}: {len(pool)} ứng viên (top {frac:.0%} vốn hóa) -> {len(tab)} mã sau xếp hạng Sortino")
    screens.append(tab)
universe = pd.concat(screens)
tickers_binance = extra_tickers + [b + quote_asset for b in universe.index]
# mã niêm yết SAU start_date: chỉ tải file từ ngày niêm yết (mã cũ hơn giữ nguyên, kể cả cho lịch sử MDD nhiều năm)
symbol_start = {b + quote_asset: str(f) for b, f in universe["first"].items() if str(f) > start_str}
print(f"Tickers: {len(tickers_binance)} (extra {len(extra_tickers)} + coin {len(screens[0])} + cổ phiếu {len(screens[1])}), {len(symbol_start)} mã niêm yết sau {start_str}")

def download_range(interval, start, end):
    """Tháng trọn vẹn -> file monthly (~30x ít request hơn); đầu kỳ lẻ và ~5 ngày cuối (monthly chưa đăng) -> file daily."""
    first = start if start.day == 1 else start.replace(day=1) + relativedelta(months=1)
    cut = (end - timedelta(days=5)).replace(day=1)
    kw = dict(symbols=tickers_binance, intervals=[interval], destination_dir=data_dir, max_workers=download_workers, retries=download_retries, symbol_start=symbol_start)
    if first >= cut:
        monthly, daily = [], [(start, end)]
    else:
        monthly = [(first, cut - relativedelta(months=1))]
        daily = [(start, first - timedelta(days=1)), (cut, end)] if start < first else [(cut, end)]
    for s, e in monthly:
        download_binance_klines_fixed_symbols(time_period="monthly", start=s.strftime("%Y-%m"), end=e.strftime("%Y-%m"), **kw)
    for s, e in daily:
        download_binance_klines_fixed_symbols(time_period="daily", start=s.strftime("%Y-%m-%d"), end=e.strftime("%Y-%m-%d"), **kw)

download_range(timeframe, start_date, end_date)

first_real = {}                                           # {ticker: nến thật đầu tiên} trước khi ffill/bfill
price_df = load_prices(tickers_binance, start_str, end_str, timeframe, data_dir, bfill=True, first_real=first_real)

# MDD dài hạn cho absolute_equal_dd: nến mdd_timeframe, cfg['mdd_years'] năm gần nhất
mdd_start = end_date - relativedelta(years=cfg['mdd_years'])
download_range(cfg['mdd_timeframe'], mdd_start, end_date)
dd_hl = tuple(load_prices(tickers_binance, mdd_start.strftime("%Y-%m-%d"), end_str, cfg['mdd_timeframe'], data_dir, col=k)
              for k in ("High", "Low"))               # đỉnh theo High, đáy theo Low -> bắt cả sụt trong ngày
print(f"MDD lookback: {len(dd_hl[0])} nến {cfg['mdd_timeframe']} ({cfg['mdd_years']} năm)")
price_df = price_df.iloc[-(display_candles + n_window + lag_candles):]   # cắt đúng số nến cần
n_fit = min(n_window, len(price_df) - 1 - lag_candles)      # n_window là input, n_fit là giá trị thực dùng -> chạy lại cell này không bị tích lũy
print(f"{len(price_df)} candles = cửa sổ fit {n_fit} + trễ {lag_candles} + backtest {len(price_df) - n_fit - lag_candles}")
# Nhãn: mã có đoạn giá được ffill/bfill (niêm yết sau điểm đầu của dữ liệu) -> thêm * trên chart / bảng weight
group = {b + quote_asset: g for b, g in universe["group"].items()}
imputed = pd.Series({t: float((price_df.index < first_real[t]).mean()) for t in price_df.columns})   # % nến trước ngày có giá thật
LABEL = {t: t + "*" if imputed[t] > 0 else t for t in price_df.columns}
info = pd.DataFrame({"group": pd.Series({t: group.get(t, "extra") for t in price_df.columns}),
                     "giá thật từ": pd.Series({t: first_real[t].date() for t in price_df.columns}),
                     "% nến ffill/bfill": (imputed * 100).round(1), "sortino (xếp hạng)": universe["sortino"].round(2).rename(lambda b: b + quote_asset)})
print("Vũ trụ (* = có đoạn giá điền ffill/bfill):"); print(info.rename(index=LABEL).to_string())
print(f"⏱ Tải & nạp dữ liệu: {time.time() - t_load:.1f}s")

In [ ]:
# --- Chạy walk-forward (pha 1 song song theo mốc rebalance x grid max_assets, pha 2 vector hóa) + bảng kết quả ---
reset_state(*RESULT_VARS)
cfg['span'] = int(cfg['cov_span_days'] * 24 * 3600 / tf_delta(timeframe).total_seconds())     # span EWMA (candle)
grid = [m for m in max_assets_range if m <= max_assets_limit]
assert grid, f"max_assets_range {list(max_assets_range)} không có giá trị nào <= max_assets_limit={max_assets_limit}"
t_run = time.time()
returns_wf, weights_hist, n_target = walk_forward(price_df, n_fit, timeframe, cfg, grid, rebalance_every=rebalance_every,
                                                  fee=fee, threshold=rebalance_threshold, lag=lag_candles, dd_hl=dd_hl, n_jobs=n_jobs, max_move=max_move)
over = [n for n, k in n_target.items() if k > max_assets_limit]    # strategy có target nắm > trần mã (đang dịch dần có thể tạm giữ nhiều mã hơn)
returns_wf, weights_hist = returns_wf.drop(columns=over), {n: w for n, w in weights_hist.items() if n not in over}
if over:
    print(f"Bỏ {len(over)} strategy vượt trần {max_assets_limit} mã: {over}")
summary = summarize(returns_wf, timeframe, weights_hist)
print(f"⏱ Walk-forward (grid {grid}, n_jobs={n_jobs}): {time.time() - t_run:.1f}s")
(summary * [100, 100, 1, 1, 100, 1]).round(2)   # cột: return_cagr%, volatility%, sharpe, sortino, max_drawdown%, turnover/năm

In [ ]:
# --- Plot: equity của mọi strategy (net of fees) ---
fig = go.Figure()
for name, r in returns_wf.items():
    fig.add_trace(go.Scatter(x=r.index, y=(1 + r).cumprod(), mode="lines", name=name,
                             line=dict(width=3 if name.startswith(("opt_", "hold_")) else 1)))
fig.update_layout(title="Walk-forward equity (net of fees)", hovermode="x unified")
fig.show()

In [ ]:
# --- Strategy tốt nhất theo 1 chỉ số: weight hiện tại + 1 chart gộp (equity = đỉnh stack, weight = độ dày từng lớp) ---
def simulate_account(metric, init_balance, min_order, dca_amount=0.0, dca_days=30):
    """Mô phỏng tài khoản theo strategy tốt nhất `metric`. Mỗi mốc rebalance, lệnh có |target - hiện tại| < min_order ($) bị bỏ.
    DCA: cứ mỗi `dca_days` ngày nạp thêm `dca_amount` ($) và rebalance ngay về target hiện hành (dca_amount = 0 -> tắt).
    Trả (tên strategy, giá trị $ từng mã + CASH theo thời gian, tổng phí, số lệnh khớp, số lệnh bị bỏ, tổng vốn đã nạp theo thời gian)."""
    best = summary[metric].idxmax()
    wh = weights_hist[best].reindex(columns=price_df.columns, fill_value=0.0)     # target weight tại các mốc rebalance
    idx = returns_wf[best].dropna().index
    r = price_df.pct_change().fillna(0).loc[idx].to_numpy()
    tw = dict(zip(idx.get_indexer(wh.index), wh.to_numpy()))                      # candle i -> target weight (áp trước return của candle i)
    hold, cash = np.zeros(r.shape[1]), init_balance                               # vị thế ($) từng mã, tiền mặt USDT (không sinh lãi)
    H, C, fees, done, skipped = np.empty(r.shape), np.empty(len(idx)), 0.0, 0, 0
    every = max(1, round(dca_days * 86400 / tf_delta(timeframe).total_seconds()))   # chu kỳ DCA, số candle
    dep = np.zeros(len(idx))
    if dca_amount > 0:
        dep[every::every] = dca_amount                                            # nạp đầu candle, trước return của candle đó
    target = None
    for i, ri in enumerate(r):
        cash += dep[i]
        target = tw.get(i, target)                                                # target mới nhất đã biết
        if i in tw or dep[i] > 0:
            d0 = target * (cash + hold.sum()) - hold                               # $ cần mua (+) / bán (-)
            d = np.where(np.abs(d0) < min_order, 0.0, d0)
            sell = -np.minimum(d, 0)
            cash += sell.sum() * (1 - fee); fees += sell.sum() * fee              # bán trước để có tiền mua
            buy = np.maximum(d, 0)
            buy *= min(1.0, cash / (buy.sum() * (1 + fee))) if buy.sum() > 0 else 1.0   # không đủ tiền -> giảm đều
            buy[buy < min_order] = 0.0
            cash -= buy.sum() * (1 + fee); fees += buy.sum() * fee
            hold += buy - sell
            done += int(((buy > 0) | (sell > 0)).sum()); skipped += int(((np.abs(d0) > 1e-9) & (buy == 0) & (sell == 0)).sum())
        hold *= 1 + ri
        H[i], C[i] = hold, cash
    values = pd.DataFrame(H, index=idx, columns=price_df.columns)
    values = values.loc[:, values.max() > 1e-9 * max(init_balance, 1)]            # bỏ mã chưa từng nắm giữ
    values["CASH"] = C
    return best, values, fees, done, skipped, pd.Series(init_balance + dep.cumsum(), index=idx)

def plot_stack(values, title):
    """Stacked area $ theo mã: đỉnh stack = equity, độ dày từng lớp = vị thế (weight). Đường đen = equity (hover hiện tổng)."""
    values = values.rename(columns=LABEL)
    fig = go.Figure([go.Scatter(x=values.index, y=values[c], stackgroup="v", name=c, line=dict(width=0.5)) for c in values.columns])
    fig.add_trace(go.Scatter(x=values.index, y=values.sum(axis=1), name="Equity", line=dict(color="black", width=1.5)))
    return fig.update_layout(title=title, hovermode="x unified")

def plot_weights(wh, title):
    """Stacked area TARGET weight (%) của strategy, đúng giá trị đã làm tròn weight_step tại mỗi lần rebalance (giữ nguyên đến lần sau).
    Weight trôi theo giá nằm ở chart equity & weights."""
    wh = wh.rename(columns=LABEL)
    wh = wh.loc[:, (wh > 0).any()] * 100
    fig = go.Figure([go.Scatter(x=wh.index, y=wh[c], stackgroup="w", name=c, line=dict(width=0.5, shape="hv")) for c in wh.columns])
    return fig.update_layout(title=title, hovermode="x unified", yaxis=dict(ticksuffix="%", range=[0, 100]))

def show_best(metric):
    best = summary[metric].idxmax()
    print(f"Best {metric}: {best} ({summary.loc[best, metric]:.4g})")
    cur = weights_hist[best].iloc[-1].rename(index=LABEL)
    print("Weight hiện tại (lần rebalance gần nhất):")
    print((cur[cur > 0].sort_values(ascending=False) * 100).round(2).astype(str) + "%")
    _, values, *_ = simulate_account(metric, 1.0, 0.0)                            # vốn 1, không giới hạn lệnh tối thiểu
    plot_stack(values, f"Equity & weights - {best} (vốn = 1)").show()
    plot_weights(weights_hist[best], f"Target weights (làm tròn theo weight_step) - {best}").show()

show_best("return_cagr")

In [ ]:
show_best("sortino")

In [ ]:
# --- Mô phỏng tài khoản thật theo strategy tốt nhất: vốn ban đầu + lệnh tối thiểu mỗi vị thế + DCA, so với gửi earn ---
sim_metric = "sortino"      # chỉ số chọn strategy (giống show_best)
init_balance = 400.0        # Vốn ban đầu (USDT ~ USD)
min_order = 5.0             # Giá trị lệnh tối thiểu ($): chênh lệch target - hiện tại nhỏ hơn mức này thì KHÔNG giao dịch
dca_amount = 50.0           # Số tiền nạp thêm mỗi kỳ DCA ($); 0 = tắt DCA
dca_days = 30               # Chu kỳ DCA (ngày). Mỗi kỳ nạp xong rebalance ngay về target hiện hành
earn_apr = 0.07             # Benchmark gửi earn: lãi suất danh nghĩa/năm, lãi kép theo ngày (mỗi ngày nhân 1 + apr/365), cùng vốn đầu + cùng dòng DCA. 0 = tắt

best, values, fees, done, skipped, invested = simulate_account(sim_metric, init_balance, min_order, dca_amount, dca_days)
eq_sim = values.sum(axis=1)
# equity lý tưởng: cùng dòng tiền DCA nhưng weight liên tục (không giới hạn lệnh tối thiểu), net of fees như backtest
r_net, dep = returns_wf[best].dropna().to_numpy(), invested.diff().fillna(0).to_numpy()
eq_ideal, e = np.empty(len(r_net)), init_balance
for i, ri in enumerate(r_net):
    e = (e + dep[i]) * (1 + ri); eq_ideal[i] = e
eq_ideal = pd.Series(eq_ideal, index=invested.index)
cpd = max(1, round(86400 / tf_delta(timeframe).total_seconds()))     # số candle / ngày
eq_earn, e = np.empty(len(dep)), init_balance
for i in range(len(dep)):
    e += dep[i]
    if i > 0 and i % cpd == 0:
        e *= 1 + earn_apr / 365                                       # trả lãi cuối mỗi ngày, nhập gốc
    eq_earn[i] = e
eq_earn = pd.Series(eq_earn, index=invested.index)
n_dca = int((dep > 0).sum())
print(f"Best {sim_metric}: {best} | vốn {init_balance:.0f}$ + DCA {dca_amount:.0f}$/{dca_days} ngày ({n_dca} lần), lệnh tối thiểu {min_order:.0f}$, phí {fee:.2%}")
print(f"Tổng vốn nạp: {invested.iloc[-1]:.2f}$ | Số dư cuối: {eq_sim.iloc[-1]:.2f}$ | lãi/lỗ: {eq_sim.iloc[-1] - invested.iloc[-1]:+.2f}$ ({eq_sim.iloc[-1] / invested.iloc[-1] - 1:+.2%})")
print(f"Lý tưởng (weight liên tục): {eq_ideal.iloc[-1]:.2f}$ ({eq_ideal.iloc[-1] / invested.iloc[-1] - 1:+.2%} trên vốn nạp)")
print(f"Gửi earn {earn_apr:.1%}/năm (kép ngày): {eq_earn.iloc[-1]:.2f}$ ({eq_earn.iloc[-1] - invested.iloc[-1]:+.2f}$, {eq_earn.iloc[-1] / invested.iloc[-1] - 1:+.2%}) | strategy hơn earn: {eq_sim.iloc[-1] - eq_earn.iloc[-1]:+.2f}$")
print(f"Lệnh đã khớp: {done} | bị bỏ vì < {min_order:.0f}$: {skipped} | tổng phí: {fees:.2f}$ | tiền mặt: {values['CASH'].iloc[-1]:.2f}$")
pos_usd = values.iloc[-1][lambda s: s > 0.01].sort_values(ascending=False).rename(index=LABEL)
print("Vị thế hiện tại:"); print(pd.DataFrame({"USD": pos_usd.round(2), "%": (pos_usd / eq_sim.iloc[-1] * 100).round(2)}).to_string())
fig = plot_stack(values, f"Equity & weights ($) - {best}")
fig.add_trace(go.Scatter(x=eq_ideal.index, y=eq_ideal, name="Equity lý tưởng (weight liên tục)", line=dict(color="black", width=1, dash="dot")))
fig.add_trace(go.Scatter(x=eq_earn.index, y=eq_earn, name=f"Gửi earn {earn_apr:.1%}/năm", line=dict(color="purple", width=1.5)))
fig.add_trace(go.Scatter(x=invested.index, y=invested, name="Tổng vốn nạp", line=dict(color="grey", width=1.5, dash="dash", shape="hv")))
fig.show()
plot_weights(weights_hist[best], f"Target weights (làm tròn theo weight_step) - {best}").show()

In [ ]:
end_time = time.time()
elapsed_seconds = end_time - start_time

# Chuyển sang định dạng hh:mm:ss
hours, rem = divmod(elapsed_seconds, 3600)
minutes, seconds = divmod(rem, 60)

print(f"Thời gian xử lý: {int(hours):02d}:{int(minutes):02d}:{seconds:05.2f}")